In [18]:
import pathlib

import numpy as np
import scipy.sparse as sp
import scipy.io
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import networkx as nx
import utils.preprocess
from sklearn.model_selection import train_test_split

save_prefix = r'F:\Data\OHNN\data\Freebase_processed\graph_split/'
# save_prefix = r'D:\OneDrive\PhD\毕业\OHNN\demo_data\DBLP_processed/'
# read_perfix = 'demo_data/'
read_perfix = r'F:\Data\graph_data\HINormer\Freebase/'
num_ntypes = 4

## load

In [19]:
node = pd.read_csv(read_perfix + 'node.dat', sep = '\t\t', header=None)
link = pd.read_csv(read_perfix + 'link.dat', sep = '\t', header=None)
meta = pd.read_csv(read_perfix + 'meta.dat', sep = '\t', header=None)
label = pd.read_csv(read_perfix + 'label.dat', sep = '\t', header=None)
# edge = pd.read_csv(read_perfix + 'edge.txt', sep = ' ', header=None) # edge is link

d:\anaconda3\envs\ohnn\lib\site-packages\pandas\util\_decorators.py:311: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  return func(*args, **kwargs)


In [20]:
meta

,0
0,Node Total: Count 43854
1,Node Type_0: Count 3492
2,Node Type_1: Count 33401
3,Node Type_2: Count 2502
4,Node Type_3: Count 4459
5,Node type:
6,movie 0
7,actor 1
8,direct 2
9,writer 3


In [21]:
link.head(11)

,0,1,2,3
0,0,3492,0,1.0
1,3492,0,1,1.0
2,0,3493,0,1.0
3,3493,0,1,1.0
4,0,3494,0,1.0
5,3494,0,1,1.0
6,0,3495,0,1.0
7,3495,0,1,1.0
8,0,3496,0,1.0
9,3496,0,1,1.0


In [22]:
np.unique(link[2])

array([0, 1, 2, 3, 4, 5], dtype=int64)

## label

In [23]:
label.shape

(1047, 4)

In [24]:
label.head(10)

,0,1,2,3
0,2175,NaN,0,0
1,1180,NaN,0,1
2,3141,NaN,0,1
3,3164,NaN,0,0
4,2099,NaN,0,0
5,3333,NaN,0,1
6,3341,NaN,0,2
7,3082,NaN,0,1
8,3185,NaN,0,0
9,777,NaN,0,2


In [25]:
[np.unique(label[p]) for p in label.columns]

[array([   1,    3,    8, ..., 3481, 3482, 3489], dtype=int64),
 array([nan]),
 array([0], dtype=int64),
 array([0, 1, 2], dtype=int64)]

In [26]:
label[[0,3]]

,0,3
0,2175,0
1,1180,1
2,3141,1
3,3164,0
4,2099,0
...,...,...
1042,3206,0
1043,2076,0
1044,2532,2
1045,857,2


In [27]:
# author2id = pd.read_csv(read_perfix + 'author2id2.txt', sep = '\t', header=None)
# conf2id = pd.read_csv(read_perfix + 'conf2id.txt', sep = '\t', header=None)
# label2id = pd.read_csv(read_perfix + 'label2id.txt', sep = '\t', header=None)
# paper_author = pd.read_csv(read_perfix + 'paper_author.txt', sep = '\t', header=None)
# paper_conference = pd.read_csv(read_perfix + 'paper_conference.txt', sep = '\t', header=None)
# paper_label = pd.read_csv(read_perfix + 'paper_label.txt', sep = '\t', header=None)
# paper_type = pd.read_csv(read_perfix + 'paper_type.txt', sep = '\t', header=None)
# paper_year = pd.read_csv(read_perfix + 'paper_year.txt', sep = '\t', header=None)


# data = [author2id, conf2id, label2id, paper_author, paper_conference, paper_label, paper_type, paper_year]
# [p.shape for p in data]

## type_mask

In [28]:
edge0 = link[link[2] == 0]
edge0.shape #(65341, 4)
edge1 = link[link[2] == 1]
edge1.shape #(65341, 4)
edge2 = link[link[2] == 2]
edge2.shape #(3762, 4)
edge3 = link[link[2] == 3]
edge3.shape #(3762, 4)
edge4 = link[link[2] == 4]
edge4.shape #(6414, 4)
edge5 = link[link[2] == 5]
edge5.shape #(6414, 4)

(6414, 4)

In [29]:
tmp = [max(link[link[2] == p][0]) for p in range(6)]
tmp

[3491, 36892, 3491, 39394, 3491, 43853]

In [30]:
tmp2 = [min(link[link[2] == p][0]) for p in range(6)]
tmp2

[0, 3492, 0, 36893, 0, 39395]

In [31]:
tmp3 = [np.unique(link[link[2] == p][0]).shape for p in range(6)]
tmp3

[(3492,), (33401,), (3492,), (2502,), (3492,), (4459,)]

In [32]:
edge2.head(11)

,0,1,2,3
130682,0,36893,2,1.0
130684,1,36894,2,1.0
130686,2,36895,2,1.0
130688,3,36896,2,1.0
130690,4,36897,2,1.0
130692,5,36898,2,1.0
130694,6,36899,2,1.0
130696,7,36900,2,1.0
130698,8,36901,2,1.0
130700,9,36902,2,1.0


In [33]:
raw_nums = [3492, 33401, 2502, 4459]
nums = sum(raw_nums)
print(raw_nums)
print(nums)

prefix_operator = np.ones((num_ntypes+1, num_ntypes))
prefix_operator = np.tril(prefix_operator, k=-1)   # 下三角矩阵
prefix_operator = prefix_operator.dot(raw_nums).astype(int)
print(prefix_operator)

# 0 for movies 4661, 1 for directors 2270, 2 for actors 5841
type_mask = np.zeros(nums,dtype=int)
for i in range(num_ntypes):
    type_mask[prefix_operator[i]:prefix_operator[i+1]] = i

np.save(save_prefix + 'prefix_operator.npy',prefix_operator)
np.save(save_prefix + 'type_mask.npy',type_mask)

[3492, 33401, 2502, 4459]
43854
[    0  3492 36893 39395 43854]


## adjM

In [17]:
adjM = sp.lil_matrix((nums,nums)) # 43854
for head,tail,_,__ in link.values:
    adjM[head,tail] = 1

scipy.sparse.save_npz(save_prefix + 'adjM.npz', adjM.tocsr())
# lil matrix cost 4s on 3700x platform

KeyboardInterrupt: 

## edges for MHGCN

In [34]:
edges = [sp.lil_matrix((nums,nums)) for _ in range(6)] # 6 edges
for i in range(6):
    edge = link[link[2] == i]
    edges[i][edge[0],edge[1]] = 1

scipy.io.savemat(save_prefix + 'A.mat', {'edges': [edge.tocsr() for edge in edges]})
edges

[<43854x43854 sparse matrix of type '<class 'numpy.float64'>'
 	with 65341 stored elements in List of Lists format>,
 <43854x43854 sparse matrix of type '<class 'numpy.float64'>'
 	with 65341 stored elements in List of Lists format>,
 <43854x43854 sparse matrix of type '<class 'numpy.float64'>'
 	with 3762 stored elements in List of Lists format>,
 <43854x43854 sparse matrix of type '<class 'numpy.float64'>'
 	with 3762 stored elements in List of Lists format>,
 <43854x43854 sparse matrix of type '<class 'numpy.float64'>'
 	with 6414 stored elements in List of Lists format>,
 <43854x43854 sparse matrix of type '<class 'numpy.float64'>'
 	with 6414 stored elements in List of Lists format>]

## ontology

In [39]:
prefix_operator = np.load(save_prefix + 'prefix_operator.npy')
type_mask = np.load(save_prefix + 'type_mask.npy')
adjM = scipy.sparse.load_npz(save_prefix + 'adjM.npz')
ontology = {
    'stem':[1,0,2],
    'branch':{0:[0,3]}
}
ontology_pairs = [[0,1],[0,2],[0,3]]

In [40]:
link_intances = utils.preprocess.get_intances_by_pair(adjM, type_mask, ontology, prefix_operator)
#print(link_intances)
print('=======done=======')

# cost about 0s with sparse matrix csr 
# nodes 312776
# stem 70902
# branch 6414

stem searching starts!
Sat Feb 24 14:48:14 2024, instances of (0, 2) have been found, counts is 3762..
merging path...

branch0 searching starts!
Sat Feb 24 14:48:14 2024, instances of (0, 3) have been found, counts is 6414.
merging path...

=======done=======


In [41]:
link_intances.keys()

dict_keys(['stem', 'branch'])

In [43]:
link_intances['branch'][0].shape

(6414, 2)

In [44]:
ontology = {
    'stem':[1,0,2],
    'branch':{0:[0,3]}
}

subgraphs = utils.preprocess.get_ontology_subgraphs_v3(ontology, link_intances)

subgraphs = subgraphs[subgraphs.columns.sort_values()]
print(len(subgraphs))
print(subgraphs)

# 0s
# 143968 subgraphs

143968
         0      1      2      3
0      672   5630  37066  40546
1      672   5637  37066  40546
2      672   4345  37066  40546
3      672   9280  37066  40546
4      672  10291  37066  40546
...    ...    ...    ...    ...
9771  1222  19228  38012  41300
9772  1222  19228  38012  41301
9773  1222  19229  38012  40143
9774  1222  19229  38012  41300
9775  1222  19229  38012  41301

[143968 rows x 4 columns]


In [45]:
subgraphs.columns

Int64Index([0, 1, 2, 3], dtype='int64')

## search incomplete, unfinished

In [46]:
ontology_pairs = [[0,1],[0,2],[0,3]]
res_adj = utils.preprocess.find_res_adj2(adjM, subgraphs)
# 55s

In [47]:
incomplete_ontology_subgraphs, incomplete_subgraphs = utils.preprocess.find_incomplete_subgraph(adjM, type_mask, ontology_pairs, res_adj)
print(len(incomplete_ontology_subgraphs))
print(incomplete_subgraphs)
# 35min unfinished on 3700x

Sat Feb 24 14:49:27 2024, finding pairs...
Sat Feb 24 14:49:28 2024, finding pairs...
Sat Feb 24 14:49:28 2024, finding pairs...
0
[]


## save

In [48]:
# create the directories if they do not exist
for i in ['complete','incomplete']:
    pathlib.Path(save_prefix + '{}'.format(i)).mkdir(parents=True, exist_ok=True)

# save data 

# mapping of node to subgraphs

# mapping of node to node pairs 

# save schema
np.save(save_prefix + 'complete/ontology.npy', ontology) # schema
np.save(save_prefix + 'ontology_pairs.npy', ontology_pairs)
# subgraph table
np.save(save_prefix + 'complete/subgraphs.npy', subgraphs)
# all nodes adjacency matrix
scipy.sparse.save_npz(save_prefix + 'adjM.npz', adjM)
# all nodes features one-hot
for i in range(num_ntypes):
    scipy.sparse.save_npz(save_prefix + 'features_{}.npz'.format(i), scipy.sparse.eye(raw_nums[i]).tocsr())
# all nodes (authors, papers, terms and conferences) type labels
np.save(save_prefix + 'node_types.npy', type_mask)
# type prefix
np.save(save_prefix + 'prefix_operator.npy', prefix_operator)
# paper labels
np.save(save_prefix + 'labels.npy', label[[0,3]]) # 3 class

### generate torch.sparse feature

## split

In [51]:
# subgraphs train/validation/test splits
rand_seed = 33333333
train_val_idx, test_idx = train_test_split(np.arange(adjM.shape[0]), test_size=0.1, random_state=rand_seed)
a = np.isin(subgraphs,test_idx)
a = a.sum(axis=1).astype('bool')
subgraphs_test = subgraphs[a]
subgraphs_tr_val = subgraphs[~a]
subgraphs[a].shape
print(subgraphs_test.shape[0]/len(subgraphs)) # 30% for test
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.04, random_state=rand_seed)
b = np.isin(subgraphs_tr_val,val_idx)
b = b.sum(axis=1).astype('bool')
subgraphs_val = subgraphs_tr_val[b]
subgraphs_train = subgraphs_tr_val[~b]
subgraphs_tr_val[b].shape
print(subgraphs_val.shape[0]/len(subgraphs)) # 10% for val
print(len(subgraphs_train)/len(subgraphs)) # 60% for train

np.savez(save_prefix + 'complete/' + 'subgraphs_train_val_test.npz',
         subgraphs_train=subgraphs_train,
         subgraphs_val=subgraphs_val,
         subgraphs_test=subgraphs_test) # subgraph train&val&test
# node split
np.savez(save_prefix + 'complete/' + 'train_val_test_nodes.npz',
         train_nodes=train_idx,
         val_nodes=val_idx,
         test_nodes=test_idx) # nodes train&val&test

0.30850605690153365
0.10045982440542343
0.5910341186930429


In [ ]:
mask = np.ones_like(adjM, dtype=bool)
for row in subgraphs.values:
    mask[np.ix_(row,row)] = False

In [ ]:
subgraphs.values[1]

In [ ]:
adjM[9]

In [ ]:
adjM[np.ix_([9,10,11,12,13,14,15], [79,80,81,82,83,84,85,86,87])]

In [ ]:
adjM[[9,10,11,12,13,14,15]][:,[79,80,81,82,83,84,85,86,87]]

In [ ]:
np.ix_([9,10,11,12,13,14,15], [79,80,81,82,83,84,85,86,87])

In [ ]:
((adjM[9] != 0))